# 📋 Guía de Verificación Completa del Pipeline

## ✅ Estructura Recreada

### 📁 Notebooks DDL (Ejecutar la primera vez)

1. **[DDL_Bronze](#notebook-3465291481176724)** (este notebook)
   * Crea: workspace.tp_dnrpa_bronze (esquema)
   * Crea: bronze_transferencias (23 columnas)
   * Crea: tabla_historica_valores_automotor

2. **[DDL_Silver](#notebook-3465291481176725)**
   * Crea: workspace.tp_dnrpa_silver (esquema)
   * Crea: silver_transferencias (24 columnas con DATE/TIMESTAMP)

3. **[DDL_Gold](#notebook-3465291481176726)**
   * Crea: workspace.tp_dnrpa_gold (esquema)
   * Crea: dim_marca, dim_tipo_vehiculo, dim_modelo, dim_geografia
   * Crea: fact_transferencias
   * Crea: vw_transferencias_analitica (vista desnormalizada)

---

### 🔄 Notebooks ETL (Ejecutar para cargar datos)

1. **[02_Bronze_dnrpa](#notebook-860821664547612)**
   * ✅ Usa INSERT OVERWRITE (no CREATE OR REPLACE)
   * Carga archivos CSV → bronze_transferencias
   * Carga archivo Excel → tabla_historica_valores_automotor

2. **[02_Silver_dnrpa](#notebook-860821664547610)**
   * Transformaciones PySpark
   * .saveAsTable() → silver_transferencias
   * ⚠️ IMPORTANTE: Filtro corregido en celda 8: `F.col("tramite") == "TRANSFERENCIA"`

3. **[04_Gold](#notebook-860821664547601)**
   * ✅ Usa INSERT OVERWRITE (no CREATE OR REPLACE)
   * Carga 4 dimensiones + 1 fact
   * 5 validaciones de calidad incluidas

---

## 🧪 Plan de Verificación Paso a Paso

### Paso 1: Crear Infraestructura (Primera vez)

```
1. Ejecutar: DDL_Bronze (todas las celdas)
2. Ejecutar: DDL_Silver (todas las celdas)
3. Ejecutar: DDL_Gold (todas las celdas)
```

**Verificación**: Ir a Databricks Explorer → Confirmar que existen:
* workspace.tp_dnrpa_bronze (con 2 tablas)
* workspace.tp_dnrpa_silver (con 1 tabla)
* workspace.tp_dnrpa_gold (con 5 tablas + 1 vista)

---

### Paso 2: Cargar Datos Bronze

```
Ejecutar: 02_Bronze_dnrpa (Run All)
```

**Verificación esperada**:
* Celda 2: "num_affected_rows": ~3.3M
* Celda 3: "num_affected_rows": depende del archivo Excel
* Celda 5: SELECT debe mostrar 100 filas de datos

---

### Paso 3: Transformar y Cargar Silver

```
Ejecutar: 02_Silver_dnrpa (Run All)
```

**Verificación esperada**:
* Celda 23: saveAsTable() debe completar sin errores
* Al final: Consultar `SELECT COUNT(*) FROM workspace.tp_dnrpa_silver.silver_transferencias`
* **Resultado esperado**: ~2.4M registros (filtrados por TRANSFERENCIA)

⚠️ **Si Silver está vacía**: Verificar celda 8 tiene el filtro correcto:
```python
df_silver = df_silver.filter(F.col("tramite") == "TRANSFERENCIA")
```

---

### Paso 4: Cargar Gold y Validar

```
Ejecutar: 04_Gold (Run All)
```

**Verificación esperada**:

**Celda 9 (Validación 1 - Conteo de Registros)**
```
dim_geografia           846
dim_marca               700
dim_modelo            10,389
dim_tipo_vehiculo       609
fact_transferencias  2,418,741
```

**Celda 10 (Validación 2 - Unicidad de PKs)**
* Todas deben mostrar "✓ OK" (excepto dim_geografia puede tener duplicados menores)

**Celda 11 (Validación 3 - Nulos en PKs/FKs)**
* Todas deben mostrar "✓ OK" (0 nulos)

**Celda 12 (Validación 4 - Integridad Referencial)**
* Todas deben mostrar "✓ OK" (0 registros huérfanos)

**Celda 13 (Validación 5 - Reconciliación)**
* registros_silver = registros_gold = 2,418,741
* porcentaje_carga = 100.00
* resultado = "✓ COINCIDENCIA EXACTA"

---

## 🎯 Verificación de la Vista Analítica

Ejecutar esta query en cualquier notebook SQL:

```sql
SELECT 
  automotor_marca_descripcion,
  registro_seccional_provincia,
  YEAR(tramite_fecha) as anio,
  COUNT(*) as total_transferencias
FROM workspace.tp_dnrpa_gold.vw_transferencias_analitica
GROUP BY 1, 2, 3
ORDER BY 4 DESC
LIMIT 10;
```

**Resultado esperado**: Top 10 de transferencias por marca/provincia/año

---

## ❌ Problemas Comunes

### Problema: "Table not found"
**Solución**: Ejecutar el notebook DDL correspondiente primero

### Problema: Silver vacía (0 registros)
**Causa**: Filtro incorrecto en 02_Silver_dnrpa celda 8
**Solución**: Cambiar filtro a `F.col("tramite") == "TRANSFERENCIA"`

### Problema: "CAST_INVALID_INPUT" en fechas
**Solución**: Ya corregido - usa TRY_CAST en celda 20 de 02_Silver_dnrpa

### Problema: Gold vacía
**Causa**: Silver vacía (ver problema anterior)
**Solución**: Corregir Silver primero, luego reejecutar Gold

---

## 🏁 Criterios de Éxito

✅ Todos los notebooks DDL ejecutan sin errores  
✅ Bronze: ~3.3M registros cargados  
✅ Silver: ~2.4M registros (filtrados por TRANSFERENCIA)  
✅ Gold: 5 tablas pobladas correctamente  
✅ Todas las validaciones muestran "✓ OK"  
✅ Vista analítica devuelve resultados  
✅ Sin errores de CAST, division by zero, o table not found  

---

## 📊 Resultado Final Esperado

```
Bronze:  3.3M registros crudos
   ↓
Silver:  2.4M registros limpios (filtrados)
   ↓
Gold:    2.4M hechos + 12,544 dimensiones
   ↓
Vista:   Datos desnormalizados listos para BI
```

# DDL Bronze - Definiciones de Estructura

**Propósito**: Define la infraestructura de la capa Bronze (esquema y tablas).

**Ejecutar**: Solo la primera vez o cuando cambie la estructura de las tablas.

**Notas**:
* Las tablas Bronze almacenan datos crudos tal como vienen de la fuente
* Se usa inferSchema => false para preservar todos los datos como STRING
* El notebook ETL `02_Bronze_dnrpa` se encarga de CARGAR los datos (INSERT OVERWRITE)

In [0]:
%sql
-- Crear el esquema Bronze si no existe
CREATE SCHEMA IF NOT EXISTS workspace.tp_dnrpa_bronze
COMMENT 'Capa Bronze - Datos crudos sin transformación del DNRPA';

In [0]:
%sql
-- Crear tabla bronze_transferencias con esquema explícito
-- Todas las columnas son STRING para preservar datos crudos
CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_bronze.bronze_transferencias (
  tramite_fecha STRING COMMENT 'Fecha del trámite (formato original)',
  fecha_inscripcion_inicial STRING COMMENT 'Fecha de inscripción inicial',
  registro_seccional_codigo STRING COMMENT 'Código del registro seccional (PK potencial)',
  registro_seccional_descripcion STRING COMMENT 'Descripción del registro seccional',
  registro_seccional_provincia STRING COMMENT 'Provincia del registro',
  automotor_origen STRING COMMENT 'Origen del vehículo',
  automotor_anio_modelo STRING COMMENT 'Año modelo del vehículo',
  automotor_tipo_codigo STRING COMMENT 'Código de tipo de vehículo',
  automotor_tipo_descripcion STRING COMMENT 'Descripción del tipo',
  automotor_marca_codigo STRING COMMENT 'Código de marca',
  automotor_marca_descripcion STRING COMMENT 'Nombre de la marca',
  automotor_modelo_codigo STRING COMMENT 'Código de modelo',
  automotor_modelo_descripcion STRING COMMENT 'Nombre del modelo',
  automotor_uso_codigo STRING COMMENT 'Código de uso del vehículo',
  automotor_uso_descripcion STRING COMMENT 'Descripción del uso',
  titular_tipo_persona STRING COMMENT 'Tipo de persona (física/jurídica)',
  titular_domicilio_localidad STRING COMMENT 'Localidad del titular',
  titular_domicilio_provincia STRING COMMENT 'Provincia del titular',
  titular_pais_nacimiento STRING COMMENT 'País de nacimiento',
  titular_porcentaje_titularidad STRING COMMENT 'Porcentaje de titularidad',
  titular_domicilio_provincia_id STRING COMMENT 'ID de provincia',
  tramite_tipo STRING COMMENT 'Tipo de trámite',
  _rescued_data STRING COMMENT 'Datos rescatados por read_files'
) USING DELTA
COMMENT 'Tabla Bronze: datos crudos de transferencias vehiculares del DNRPA';

In [0]:
%sql
-- Crear tabla de valores históricos (Excel)
CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_bronze.tabla_historica_valores_automotor (
  marca STRING,
  modelo STRING,
  version STRING,
  anio STRING,
  precio_promedio STRING,
  _rescued_data STRING
) USING DELTA
COMMENT 'Tabla Bronze: valores históricos de vehículos (fuente Excel)';